<a href="https://colab.research.google.com/github/SJGLAB/SA-MPNN/blob/main/SA_MPNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
# 1. check the version of torch
!python -c "import torch; print('torch_version:', torch.__version__)"
# 2. install pytorch-lightning package
!pip install pytorch-lightning --no-deps
!pip install lightning-utilities torchmetrics
!pip install Bio
!pip install fair-esm

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 45.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 81.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 60.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 105.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196

In [2]:
import os
if not os.path.exists('SA-MPNN'):
    !git clone https://github.com/SJGLAB/SA-MPNN.git
%cd SA-MPNN
!mkdir -p best_model
!wget -q -O best_model/best_sa_mpnn.ckpt "https://github.com/SJGLAB/SA-MPNN/releases/download/v1.0.0/best_SA_MPNN.ckpt"

import pytorch_lightning as pl
!sed -i 's|/home/bingxing2/home/scx6a62/zxy/|./|g' local.yaml
!wget -q -O esm2_t30_150M_UR50D.pt https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t30_150M_UR50D.pt
!wget -q -O esm2_t30_150M_UR50D-contact-regression.pt https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t30_150M_UR50D-contact-regression.pt
!sed -i 's|/home/bingxing2/home/scx6a62/zxy/esm2_t30_150M_UR50D.pt|./esm2_t30_150M_UR50D.pt|g' transfer_model_self.py
!sed -i 's|/home/bingxing2/home/scx6a62/zxy/esm2_t30_150M_UR50D.pt|./esm2_t30_150M_UR50D.pt|g' custom_inference_colab.py


/content/SA-MPNN


In [18]:
# Cell 3: 🌟 SA-MPNN Structure-Guided Mutation Design Platform (Click Run to Start)
#@title Please set your structure and scanning parameters below, then click the Run button.

#@markdown ---
#@markdown ### 1. Structure Input
#@markdown Select the PDB input method: enter a PDB ID to fetch automatically, or check the box to upload a local PDB file.
PDB_ID = "1ACB" #@param {type:"string"}
Upload_Local_PDB = False #@param {type:"boolean"}
Target_Chain = "E" #@param {type:"string"}

#@markdown ### 2. Task Mode
#@markdown Select between comprehensive DMS scanning or a specific single-point mutation query.
Mode = "Single Mutation Query" #@param ["Comprehensive DMS Scan", "Single Mutation Query"]

#@markdown ### 3. Output Settings
#@markdown If running a DMS scan, set the number of top-performing mutations to display.
Show_Top_N = 50 #@param {type:"slider", min:5, max:50, step:5}
#@markdown If running a single query, enter the mutation here (e.g., C1A), otherwise leave blank.
Specific_Mutation = "C1A" #@param {type:"string"}
#@markdown ---

import os
import glob
import pandas as pd
from google.colab import files

print("========================================")
print("🧬 Initializing SA-MPNN Structural Perception Engine...")

# --- 1. Handle PDB Input ---
pdb_path = f"{PDB_ID}.pdb"
if Upload_Local_PDB:
    print("📥 Please select your .pdb file in the pop-up window:")
    uploaded = files.upload()
    pdb_path = list(uploaded.keys())[0]
    print(f"✅ Successfully loaded local structure: {pdb_path}")
else:
    print(f"🌐 Fetching structure from RCSB PDB: {PDB_ID}...")
    !wget -q -O {pdb_path} https://files.rcsb.org/download/{PDB_ID}.pdb
    print(f"✅ Successfully fetched: {pdb_path}")

print(f"🎯 Target Chain: Chain {Target_Chain}")
print("⚙️ SA-MPNN is extracting 3D features and performing high-throughput thermodynamic calculations (please wait)...\n")

# =====================================================================
# 🚀 Inference Engine: Executing SA-MPNN command
# =====================================================================
!mkdir -p ./results

# Inject UI parameters into the inference script
!python custom_inference.py \
    --pdb {pdb_path} \
    --chain {Target_Chain} \
    --model_path ./best_model/best_sa_mpnn.ckpt \
    --out_dir ./results

print("\n✅ Inference complete. Parsing data...")
# =====================================================================

# --- 2. Locate output file ---
output_csv = None
result_files = glob.glob("./results/*.csv")

if result_files:
    output_csv = max(result_files, key=os.path.getctime)
    print(f"📄 Located results file: {output_csv}")
else:
    print("❌ Error: No CSV output found in ./results directory. Please check for model errors.")

# --- 3. Result Visualization ---
if output_csv and os.path.exists(output_csv):
    df = pd.read_csv(output_csv)

    # Identify relevant columns
    ddg_col = next((col for col in df.columns if col.lower() in ["ddg_pred", "ddg", "score"]), None)
    mut_col = next((col for col in df.columns if col.lower() in ["mutation", "mut"]), None)

    if not ddg_col:
         print(f"⚠️ Warning: Could not auto-detect ddG column in {df.columns.tolist()}.")
    else:
        df_sorted = df.sort_values(by=ddg_col, ascending=True).reset_index(drop=True)

        print("========================================")
        print("🏆 SA-MPNN Prediction Finished!")

        if "DMS" in Mode:
            print(f"📊 Mode: Comprehensive DMS Scan (Total computed: {len(df)})")
            print(f"🔥 Top {Show_Top_N} most stabilizing mutations:")

            top_candidates = df_sorted.head(Show_Top_N)
            display(top_candidates)

            print("\n💾 Downloading full DMS results table...")
            files.download(output_csv)
            print(f"✅ Download started: {os.path.basename(output_csv)}")

        elif "Single" in Mode:
            if Specific_Mutation == "":
                print("❌ Error: Please enter a mutation (e.g., C1A).")
            else:
                try:
                    wt_char = Specific_Mutation[0].upper()
                    mut_char = Specific_Mutation[-1].upper()
                    pos_val = int(Specific_Mutation[1:-1])-1

                    result = df[(df['pdb_res_id'] == pos_val) &
                                (df['wildtype'] == wt_char) &
                                (df['mutation'] == mut_char)]

                    if not result.empty:
                        val = result.iloc[0][ddg_col]
                        print(f"▶️ Prediction for {Specific_Mutation}: ΔΔG = {val:.3f} kcal/mol")
                        print("💡 Conclusion: " + ("Stabilizing" if val < 0 else "Destabilizing"))
                    else:
                        print(f"⚠️ Mutation {Specific_Mutation} not found. Please check residue numbering.")
                except:
                    print("❌ Format Error: Please use format 'C1A'.")
        print("========================================")
else:
    print("❌ Prediction failed: No CSV file generated.")

🧬 Initializing SA-MPNN Structural Perception Engine...
🌐 Fetching structure from RCSB PDB: 1ACB...
✅ Successfully fetched: 1ACB.pdb
🎯 Target Chain: Chain E
⚙️ SA-MPNN is extracting 3D features and performing high-throughput thermodynamic calculations (please wait)...

/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
CUDA 可用状态: True
if
/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/py